In [1]:
from openai import OpenAI
import gradio as gr
import os
import json
import requests
import sqlite3

from dotenv import load_dotenv
from services import (
    insert_medicines_from_csv,
    get_medicine_info,
    insert_interactions_from_json,
    check_drug_interaction,
    drug_lookup,
    insert_comprehensive_drugs_from_csv,
    get_drugs_by_class,
    get_drug_details
)

from services.phase3_medicine_llm_schema import MEDICINE_TOOLS

In [5]:
# One-time database setup
insert_medicines_from_csv()

✓ Database schema created successfully!
✓ Data imported successfully!
✓ Total medicines inserted: 11825


Test Query

In [6]:
conn = sqlite3.connect('db/medicine_info.db')
cursor = conn.cursor()

# Fetch and print one record to verify data insertion
cursor.execute('''
    SELECT *
    FROM medicines
    LIMIT 1
        ''')
        
result = cursor.fetchone()

print(result)
        
# Fetch and print the total count of records in the medicines table        
cursor.execute('''
    SELECT count(*)
    FROM medicines
        ''')
        
result = cursor.fetchone()
print(result)

conn.close()

(1, 'Avastin 400mg Injection', 'Bevacizumab (400mg)', 'Roche Products India Pvt Ltd', ' Cancer of colon and rectum Non-small cell lung cancer Kidney cancer Brain tumor Ovarian cancer Cervical cancer', 'Rectal bleeding Taste change Headache Nosebleeds Back pain Dry skin High blood pressure Protein in urine Inflammation of the nose')
(11825,)


In [7]:
aspirin_medicine_info = get_medicine_info("Aspirin")

for result in aspirin_medicine_info:
    print(result)

{'medicine_name': 'Aztor Asp 75 Capsule', 'composition': 'Atorvastatin (10mg) + Aspirin (75mg)', 'manufacturer': 'Sun Pharmaceutical Industries Ltd', 'uses': 'Treatment of Prevention of heart attack and stroke', 'side_effects': 'Abdominal pain Constipation Flatulence Increased liver enzymes Hepatitis viral infection of liver Reye s syndrome like symptoms'}
{'medicine_name': 'Aztogold 20 Capsule', 'composition': 'Aspirin (75mg) + Atorvastatin (20mg) + Clopidogrel (75mg)', 'manufacturer': 'Sun Pharmaceutical Industries Ltd', 'uses': 'Prevention of Heart attack', 'side_effects': 'Increased bleeding tendency Abdominal pain Indigestion Bruise Nosebleeds Gastrointestinal bleeding Diarrhea'}
{'medicine_name': 'Atchol-ASP Capsule', 'composition': 'Atorvastatin (10mg) + Aspirin (75mg)', 'manufacturer': 'Aristo Pharmaceuticals Pvt Ltd', 'uses': 'Treatment of Prevention of heart attack and stroke', 'side_effects': 'Abdominal pain Constipation Flatulence Increased liver enzymes Hepatitis viral inf

In [2]:
aspirin_lookup_result = drug_lookup("Aspirin")
print(aspirin_lookup_result)

Inside the drug_lookup function
Querying OpenFDA API for generic name: Aspirin
Successfully retrieved data from OpenFDA API
{'brand_name': 'Low Dose Aspirin', 'generic_name': 'ASPIRIN', 'active_ingredients': ['Active ingredient (in each tablet) Aspirin 81 mg (NSAID)* *nonsteroidal anti-inflammatory drug'], 'purpose': 'Purpose Pain reliever', 'indications': 'Uses for the temporary relief of minor aches and pains or as recommended by your doctor. Because of its delayed action, this product will not provide fast relief of headaches or other symptoms needing immediate relief. ask your doctor about other uses for safety coated 81 mg aspirin', 'warnings': "Warnings Reye's syndrome : Children and teenagers who have or are recovering from chicken pox or flu-like symptoms should not use this product. When using this product, if changes in behavior with nausea and vomiting occur, consult a doctor because these symptoms could be an early sign of Reye's syndrome, a rare but serious illness. Allerg

In [ ]:
insert_interactions_from_json()
interaction_result = check_drug_interaction("Aspirin", "Warfarin")
print(interaction_result)


✓ Database schema created successfully!
✓ Drug interaction data inserted successfully!
✓ Total interactions inserted: 80
{'interaction_id': 78, 'drug_a': 'Warfarin', 'drug_b': 'Aspirin', 'severity': 'Major', 'mechanism': 'Pharmacodynamic: Additive antiplatelet effects and GI irritation.', 'clinical_effect': 'Significantly increased risk of major bleeding, especially gastrointestinal bleeding.', 'safer_alternative': 'Use only if strong indication exists (e.g., mechanical heart valve)', 'clinical_management': 'This combination should only be used when the benefit clearly outweighs the risk (e.g., after coronary stent placement). Use low-dose aspirin (81 mg) and the lowest effective warfarin dose. Monitor for bleeding and consider PPI prophylaxis.', 'reference': 'Chest Anticoagulation Guidelines'}


In [3]:
insert_comprehensive_drugs_from_csv()

✓ Database schema created successfully!
✓ Drug data inserted successfully!
✓ Total records inserted: 718


In [5]:
class_result = get_drugs_by_class("NSAID")
print(class_result)
get_drug_details_result = get_drug_details("Aspirin")
print(get_drug_details_result)

[{'generic_name': 'Diclofenac', 'drug_class': 'NSAID', 'indications': 'Pain', 'side_effects': 'inflammation', 'availability': 'Prescription'}, {'generic_name': 'Etodolac', 'drug_class': 'NSAID', 'indications': 'Osteoarthritis', 'side_effects': 'pain', 'availability': 'Prescription'}, {'generic_name': 'Flurbiprofen', 'drug_class': 'NSAID', 'indications': 'Pain', 'side_effects': 'inflammation', 'availability': 'Prescription'}, {'generic_name': 'Ibuprofen', 'drug_class': 'NSAID', 'indications': 'Pain', 'side_effects': 'fever', 'availability': 'No prescription required/Prescription'}, {'generic_name': 'Indomethacin', 'drug_class': 'NSAID', 'indications': 'Gout', 'side_effects': 'pain', 'availability': 'Prescription'}, {'generic_name': 'Ketoprofen', 'drug_class': 'NSAID', 'indications': 'Pain', 'side_effects': 'inflammation', 'availability': 'Prescription'}, {'generic_name': 'Ketorolac', 'drug_class': 'NSAID', 'indications': 'Pain (short-term management)', 'side_effects': 'GI upset', 'avail

In [8]:
mistral_model = "mistral:latest"
llama_model = "llama3.2:1b"
tiny_model = "tinyllama:latest"
neural_model = "neural-chat:latest"

# Defining openAI client for local ollama models

# client = OpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama"  # required but can be any string
# )

# Defining openAI client for OpenAI hosted models
load_dotenv()  # Load environment variables from .env file

gpt_model = "gpt-4o-mini"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# curr_model = llama_model
curr_model = gpt_model

In [3]:
system_prompt = """
You are a comprehensive pharmaceutical information assistant. Your role is to help users find accurate and reliable information about medications.

You have access to five tools:
1. get_medicine_info: Search a local database of medicines by brand or generic name. Returns medicine details including manufacturer, uses, and side effects.
2. drug_lookup: Query the OpenFDA API for official FDA drug label information including warnings, indications, and safety data.
3. check_drug_interactions: Check for drug-drug interactions between two medications. Returns severity, mechanism, clinical effects, safer alternatives, and management recommendations.
4. get_therapeutic_alternatives: Find alternative drugs in the same therapeutic class. Returns comparable medications with indications, side effects, and availability.
5. compare_drugs: Compare multiple drugs side-by-side with detailed information on indications, side effects, dosage, route of administration, availability, and contraindications.

When a user asks about medications:
1. Use the appropriate tool(s) based on their question
2. For single drug info → get_medicine_info + drug_lookup
3. For interactions → check_drug_interactions
4. For alternatives → get_therapeutic_alternatives
5. For comparisons → compare_drugs
6. Present information clearly and organized
7. Always emphasize warnings and safety information
8. Never provide medical advice—only factual information
9. Remind users to consult healthcare professionals before taking medications

Always prioritize accuracy and user safety.
"""

In [2]:
tools = MEDICINE_TOOLS
tools

[{'type': 'function',
  'function': {'name': 'get_medicine_info',
   'description': 'Find medicine information by brand name or generic name from local database. Returns up to 5 matching results with manufacturer, uses, and side effects.',
   'parameters': {'type': 'object',
    'properties': {'search_term': {'type': 'string',
      'description': "The medicine brand name or generic name to search for (e.g., 'Aspirin', 'Ibuprofen')"}},
    'required': ['search_term']}}},
 {'type': 'function',
  'function': {'name': 'drug_lookup',
   'description': 'Query the OpenFDA API for drug label information including warnings, indications, active ingredients, and side effects using the generic drug name.',
   'parameters': {'type': 'object',
    'properties': {'generic_name': {'type': 'string',
      'description': "The generic name of the drug to look up (e.g., 'Aspirin', 'acetylsalicylic acid')"}},
    'required': ['generic_name']}}},
 {'type': 'function',
  'function': {'name': 'check_drug_int

In [16]:
def handle_tool_calls(message):
    
    # print("Inside handle_tool_calls function")
    # print("Received message:", message)

    responses = []

    for tool_call in message.tool_calls:
        # print("Processing tool call:", tool_call)

        tool_result = None

        if tool_call.function.name == "drug_lookup":
            try:
                # print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            generic_name = args.get("generic_name")

            if not generic_name:
                tool_result = {"error": "No generic name provided."}
            else:   
                print(f"Calling function 'drug_lookup' with argument(s) '{generic_name}'")
                tool_result = drug_lookup(generic_name)
        
        elif tool_call.function.name == "get_medicine_info":
            try:
                # print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            search_term = args.get("search_term")

            if not search_term:
                tool_result = {"error": "No search term provided."}
            else:   
                print(f"Calling function 'get_medicine_info' with argument(s) '{search_term}'")
                tool_result = get_medicine_info(search_term)

        elif tool_call.function.name == "check_drug_interactions":
            try:
                # print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            drug_a = args.get("drug_a")
            drug_b = args.get("drug_b")

            if not drug_a or not drug_b:
                tool_result = {"error": "Both drug_a and drug_b are required."}
            else:
                print(f"Calling function 'check_drug_interactions' with argument(s) '{drug_a}', '{drug_b}'")
                tool_result = check_drug_interaction(drug_a, drug_b)
                if not tool_result:
                    tool_result = {"message": f"No known interaction between {drug_a} and {drug_b}"}

        elif tool_call.function.name == "get_therapeutic_alternatives":
            try:
                # print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            drug_name = args.get("drug_name")

            if not drug_name:
                tool_result = {"error": "No drug name provided."}
            else:
                print(f"Calling function 'get_therapeutic_alternatives' with argument(s) '{drug_name}'")
                drug_details = get_drug_details(drug_name)
                if drug_details:
                    drug_class = drug_details.get("drug_class")
                    alternatives = get_drugs_by_class(drug_class)
                    tool_result = {
                        "drug_name": drug_name,
                        "drug_class": drug_class,
                        "alternatives": alternatives
                    }
                else:
                    tool_result = {"error": f"Drug {drug_name} not found in database"}

        elif tool_call.function.name == "compare_drugs":
            try:
                # print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            drug_list = args.get("drug_list")

            if not drug_list or not isinstance(drug_list, list):
                tool_result = {"error": "drug_list must be provided as a list of drug names."}
            else:
                print(f"Calling function 'compare_drugs' with argument(s) {drug_list}")
                comparison = []
                for drug in drug_list:
                    drug_info = get_drug_details(drug)
                    if drug_info:
                        comparison.append(drug_info)
                    else:
                        comparison.append({"generic_name": drug, "error": "Drug not found"})
                tool_result = {"comparison": comparison}

        else:
            print(f"Unknown tool called: {tool_call.function.name}")
            tool_result = {"error": f"Unknown tool called: {tool_call.function.name}"}

        response = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            }

        responses.append(response)
    return responses

In [19]:
def med_tool_chat(message, history):

    # print("Inside the med_tool_chat function")
    print(f"User: {message}")  # ✅ Print user message
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        content = h["content"]
        if isinstance(content, list):
            content = content[0]["text"] if content else ""
        messages.append({
            "role": h["role"], 
            "content": content
            })
    
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=curr_model,
        messages=messages,
        temperature=0,
        tools=tools
    )

    max_iterations = 5
    iteration = 0

    while response.choices[0].message.tool_calls and iteration < max_iterations:
        iteration += 1

        assistant_message = response.choices[0].message

        messages.append({
            "role": "assistant",
            "content": assistant_message.content or "",
            "tool_calls": assistant_message.tool_calls

        })

        tool_responses = handle_tool_calls(assistant_message)
        messages.extend(tool_responses)

        # for msg in messages:
        #     print(f"{msg['role']}: {msg['content']}")

        response = client.chat.completions.create(
            model=curr_model,
            messages=messages,
            temperature=0,
            tools=tools
        )
    
    if iteration == max_iterations and response.choices[0].message.tool_calls:
        print("Max iterations reached while processing tool calls. Some tool calls may not have been handled.") 

    return response.choices[0].message.content or ""

In [20]:
gr.ChatInterface(fn=med_tool_chat).launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


User: Hey
User: Tell me about Aspirin
Calling function 'get_medicine_info' with argument(s) 'Aspirin'
Calling function 'drug_lookup' with argument(s) 'acetylsalicylic acid'
Inside the drug_lookup function
Querying OpenFDA API for generic name: acetylsalicylic acid
Failed to retrieve data from OpenFDA API. Status code: 404
User: What if I intake Aspirin and Ibuprofen together?
Calling function 'check_drug_interactions' with argument(s) 'Aspirin', 'Ibuprofen'
User: what is the drug class of Ibuprofen?
Calling function 'get_medicine_info' with argument(s) 'Ibuprofen'
User: give me more details regarding Aspirin
Calling function 'get_medicine_info' with argument(s) 'Aspirin'
Calling function 'drug_lookup' with argument(s) 'Acetylsalicylic Acid'
Inside the drug_lookup function
Querying OpenFDA API for generic name: Acetylsalicylic Acid
Failed to retrieve data from OpenFDA API. Status code: 404
